# vicuna-13b가 DB 환경의 출력 포맷(`Action: Operation`)을 지키는지 빠른 확인

Colab에서 *런타임 유형: GPU (T4)*로 실행하세요.

이건 `project_db`의 공식 4개 모델 비교에는 포함되지 않은 **별도의 빠른 진단용 노트북**입니다.
`llama-3.1-8b`가 DB 환경에서 지시된 `Action: Operation` 문구를 못 지켜 완료율 0%가 나왔는데,
더 오래되고(2023년) 도구사용 훈련이 안 된 `vicuna-13b`도 같은 문제가 있는지 SQL 문제 3개(SELECT/INSERT/UPDATE
각 1개)에 대해 실제 응답을 눈으로 확인합니다. 저장소 클론 없이 이 파일 하나로 바로 실행됩니다.

In [ ]:
!pip install -q transformers accelerate bitsandbytes

In [ ]:
import torch
print("GPU 사용 가능:", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "(없음, CPU로 진행 시 매우 느릴 수 있음)")

In [ ]:
# THUDM/AgentBench src/server/tasks/dbbench/__init__.py의 big_prompt 원문 그대로
SYSTEM_PROMPT = """
I will ask you a question, then you should help me operate a MySQL database with SQL to answer the question.
You have to explain the problem and your solution to me and write down your thoughts.
After thinking and explaining thoroughly, every round you can choose to operate or to answer.
your operation should be like this:
Action: Operation
```sql
SELECT * FROM table WHERE condition;
```
You MUST put SQL in markdown format without any other comments. Your SQL should be in one line.
Every time you can only execute one SQL statement. I will only execute the statement in the first SQL code block. Every time you write a SQL, I will execute it for you and give you the output.
If you are done operating, and you want to commit your final answer, then write down:
Action: Answer
Final Answer: ["ANSWER1", "ANSWER2", ...]
DO NOT write this pattern unless you are sure about your answer. I expect an accurate and correct answer.
Your answer should be accurate. Your answer must be exactly the same as the correct answer.
If the question is about modifying the database, then after done operation, your answer field can be anything.
If your response cannot match any pattern I mentioned earlier, you will be judged as FAIL immediately.
Your input will be raw MySQL response, you have to deal with it by yourself.
"""

# project_db/data/sample_cases.jsonl에서 타입별 1개씩 그대로 가져온 실제 문제
SAMPLES = [
    {
        "type": "other (SELECT)",
        "table_name": "US Ambassadors and Envoy Extraordinary to Colombia",
        "question": (
            "What is the Presentation of Credentials has a Termination of Mission listed as August 15, 2000?\n"
            "The name of this table is US Ambassadors and Envoy Extraordinary to Colombia, and the headers of "
            "this table are Representative,Title,Presentation of Credentials,Termination of Mission,Appointed by."
        ),
    },
    {
        "type": "INSERT",
        "table_name": "School Location Table",
        "question": (
            "Insert 'Madison High School' into 'School Location Table' with values 'San Diego' for 'Location', "
            "'1980' for 'Date moved', and 'nearby junior high school' for 'Currently at this location'.\n"
            "The name of this table is School Location Table, and the headers of this table are "
            "School,Location,Date moved,Currently at this location."
        ),
    },
    {
        "type": "UPDATE",
        "table_name": "Football Team Results",
        "question": (
            "Update the results of the football team named Ballarat FL from Creswick with 5 wins, 3 byes, "
            "8 losses, 2 draws and against a total score of 1305.\n"
            "The name of this table is Football Team Results, and the headers of this table are "
            "Ballarat FL,Wins,Byes,Losses,Draws,Against."
        ),
    },
]
print(f"{len(SAMPLES)}개 샘플 준비 완료 (SELECT/INSERT/UPDATE 각 1개)")

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_ID = "lmsys/vicuna-13b-v1.5"  # 게이트 모델 아님 -- HF 토큰/라이선스 동의 불필요
bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=bnb_config, device_map="auto")
print("모델 로드 완료")

In [ ]:
# vicuna-13b-v1.5는 tokenizer_config.json에 chat_template이 없어서
# apply_chat_template()이 "chat_template is not set" 에러를 냅니다.
# FastChat의 공식 vicuna_v1.1 템플릿으로 프롬프트를 직접 문자열로 조립합니다.
VICUNA_SYSTEM = (
    "A chat between a curious user and an artificial intelligence assistant. "
    "The assistant gives helpful, detailed, and polite answers to the user's questions."
)

for sample in SAMPLES:
    prompt = (
        f"{VICUNA_SYSTEM} USER: {SYSTEM_PROMPT.strip()} ASSISTANT: Ok.</s>"
        f"USER: {sample['question']} ASSISTANT:"
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    out = model.generate(**inputs, max_new_tokens=300, do_sample=False)
    reply = tokenizer.decode(out[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)

    print(f"===== [{sample['type']}] {sample['table_name']} =====")
    print(reply)
    print()
    print("-> 'Action: Operation' 포함 여부:", "Action: Operation" in reply)
    print("=" * 70)
    print()
